[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-feature-eng.ipynb)

# Feature Engineering

*AIBits Academy · Machine Learning End To End · Data Preparation · New*

The single highest-leverage activity in applied ML — constructing new, more informative columns from the ones you already have, using pandas' groupby, merge, and reshape tools.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **📋 Real-World Case Study — Marketing Customer Value & Engagement Segmentation**
>
> An auto-insurance marketer working with 9,134 policyholders found that only 14.3% of customers responded to renewal offers overall — but the raw columns didn't say *who* was worth targeting. Two engineered features unlocked the insight: a `CLV Segment` (High/Low, split at the median customer lifetime value of \$5,780) and a `Policy Age Segment` (High/Low, split at the median 48 months since policy inception), both built with a one-line `pandas.apply` threshold rule. Cross-tabulating the two revealed the highest-responding group wasn't high-value customers at all — it was **long-tenured, low-CLV** policyholders (16.2% response), beating high-CLV/long-tenure customers (13.9%). Without the engineered segment columns, that inversion was invisible in the raw 24-column table.

## Why Feature Engineering Beats Algorithm Choice, Often

Swapping Logistic Regression for XGBoost typically buys a few percentage points of accuracy. A well-constructed feature — one that directly encodes a business insight the raw columns don't expose — can buy far more, because it hands the model a signal it could never have discovered from raw columns alone.

## 1. Aggregation Features via GroupBy

The most common real-world pattern: you have *transaction-level* data but want to predict something at the *customer/entity level*. pandas' split-apply-combine (`groupby`) is the tool:

In [ ]:
import pandas as pd
import numpy as np

# Raw Swiggy order-level data — one row per order
np.random.seed(5)
orders = pd.DataFrame({
    'customer_id': np.random.choice(['C001','C002','C003','C004'], 400),
    'order_value': np.random.uniform(80,900,400).round(0),
    'cuisine': np.random.choice(['North Indian','South Indian','Chinese','Bakery'], 400),
    'rating_given': np.random.randint(1,6,400),
})

# Aggregate to customer-level features — this IS the feature engineering step
customer_features = orders.groupby('customer_id').agg(
    total_orders=('order_value', 'count'),
    avg_order_value=('order_value', 'mean'),
    total_spend=('order_value', 'sum'),
    order_value_std=('order_value', 'std'),      # spending consistency
    avg_rating_given=('rating_given', 'mean'),
    favourite_cuisine=('cuisine', lambda s: s.mode()[0]),  # most-ordered cuisine
).reset_index()
print(customer_features)

`order_value_std` is a genuinely new signal — "spends erratically" vs. "spends consistently" — that *cannot exist* in the raw transaction table at all. This is the essence of feature engineering: information that only becomes visible after aggregation.

## 2. Datetime Feature Extraction

A raw timestamp is nearly useless to most ML algorithms directly — but the signals hidden inside it (day of week, festival proximity, hour of day) are often highly predictive:

In [ ]:
orders_dt = pd.DataFrame({
    'order_ts': pd.to_datetime(['2026-01-14 20:15','2026-01-15 13:05',
                                '2026-03-08 21:40','2026-08-15 12:00'])
})
orders_dt['day_of_week']  = orders_dt['order_ts'].dt.day_name()
orders_dt['is_weekend']   = orders_dt['order_ts'].dt.dayofweek.isin([5,6]).astype(int)
orders_dt['hour']         = orders_dt['order_ts'].dt.hour
orders_dt['meal_slot']    = pd.cut(orders_dt['hour'], bins=[0,11,16,21,24],
                                labels=['Breakfast','Lunch','Dinner','Late Night'])
# Days-to-nearest-major-Indian-festival is a powerful demand-forecasting feature
festivals = pd.to_datetime(['2026-01-14','2026-03-06','2026-08-15','2026-10-20'])
orders_dt['days_to_festival'] = orders_dt['order_ts'].apply(
    lambda d: min(abs((d.normalize()-f).days) for f in festivals))
print(orders_dt[['day_of_week','is_weekend','meal_slot','days_to_festival']])

## Seeing Cyclical Encoding — Why Hour "Wraps Around"

Drag both handles below to compare any two hours. On the plain number line, distance is just |A−B|. On the circle (using `sin`/`cos` of the hour), distance reflects true time-of-day closeness — try setting A=23 and B=0 to see the mismatch the Q&A below describes.

## 3. Merging & Joining Multiple Sources

Real feature sets rarely come from a single table — customer demographics, transaction history, and support tickets usually live in separate sources that must be joined correctly:

In [ ]:
customers = pd.DataFrame({
    'customer_id': ['C001','C002','C003','C005'],
    'city': ['Bengaluru','Hyderabad','Pune','Chennai'],
    'signup_year': [2022,2023,2021,2024]
})
# left join keeps every row of customer_features, even C004 which has no demographic record
merged = customer_features.merge(customers, on='customer_id', how='left')
print(merged[['customer_id','city','total_spend']])
# NaN city for C004 (no demographic record), and C005 silently dropped (inner-join-only rows)

> **⚠ The Silent Join Bug**
>
> An inner join (the pandas default) silently drops any row without a match on both sides — losing training examples without an error or warning. Always check `len(df_before)` vs `len(df_after_merge)` after joining, and prefer an explicit `how='left'` when the left table defines your population of interest.

## 4. Interaction & Polynomial Features

Sometimes the *ratio* or *product* of two features carries more signal than either alone. EMI-to-income ratio predicts loan default far better than EMI amount or income individually:

In [ ]:
loans = pd.DataFrame({'emi':[15000,28000,9000], 'income':[60000,55000,50000]})
loans['emi_to_income'] = loans['emi'] / loans['income']      # ratio feature
loans['emi_income_interaction'] = loans['emi'] * loans['income']  # product feature
# sklearn can generate ALL pairwise interactions/polynomials automatically:
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
print(poly.fit_transform(loans[['emi','income']]))

## 5. Memory-Efficient Categoricals — pandas' Categorical dtype

Beyond ML encoding, storing repeated string categories as pandas' native `Categorical` type dramatically reduces memory and speeds up groupby operations — genuinely useful on large Indian e-commerce datasets with millions of rows:

In [ ]:
big = pd.DataFrame({'city': np.random.choice(
    ['Mumbai','Delhi','Bengaluru','Hyderabad','Ahmedabad'], 2_000_000)})
print(f"object dtype:      {big['city'].memory_usage(deep=True)/1e6:.1f} MB")
big['city'] = big['city'].astype('category')
print(f"category dtype:    {big['city'].memory_usage(deep=True)/1e6:.1f} MB")

## 6. Automatic Feature Selection

Once you've constructed a rich feature set, not every feature earns its place — irrelevant or redundant features add noise, slow training, and increase overfitting risk. Three standard automatic selection strategies, in increasing order of sophistication (and cost):

| Strategy | Mechanism | Trade-off |
|---|---|---|
| **Univariate** | Score each feature independently against the target (e.g., `SelectKBest` with an F-test or mutual information), keep the top k | Fast, but ignores feature interactions — a feature useless alone but powerful in combination gets discarded |
| **Model-Based** | Fit a model with built-in importance (Random Forest, Lasso) once, keep features above an importance threshold | Captures some interactions (via the model), but importance is specific to that one model type |
| **Iterative (RFE)** | Recursive Feature Elimination — repeatedly fit a model, drop the weakest feature, refit, until the target count is reached | Most thorough (accounts for interactions at each step), but expensive — refits the model once per eliminated feature |

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.ensemble import RandomForestClassifier
import numpy as np

# Reuse the Ahmedabad-apartment-style feature set: 4 real signals + 6 noise columns
np.random.seed(6)
n = 400
real = np.random.randn(n, 4)
noise = np.random.randn(n, 6)
Xf = np.column_stack([real, noise])
yf = (real[:,0] + 2*real[:,1] - real[:,2] > 0).astype(int)

# Univariate
kbest = SelectKBest(f_classif, k=4).fit(Xf, yf)
print(f"Univariate selected indices: {np.where(kbest.get_support())[0]}")

# Model-based (Random Forest importance)
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xf, yf)
top4_rf = np.argsort(rf.feature_importances_)[-4:]
print(f"Random Forest top-4 indices:  {sorted(top4_rf)}")

# Iterative RFE
rfe = RFE(RandomForestClassifier(n_estimators=100, random_state=42), n_features_to_select=4).fit(Xf, yf)
print(f"RFE selected indices:          {np.where(rfe.support_)[0]}")
# All three correctly converge on indices [0,1,2,3] — the 4 genuinely informative columns

All three methods correctly identify the same 4 genuinely informative columns here — with clean, additive signal like this, cheap univariate selection performs just as well as expensive RFE. The gap between methods widens on real data with feature interactions, correlated features, or non-linear relevance, which is exactly when the extra cost of RFE starts paying for itself.

## Feature Engineering Checklist

| Question to ask | Technique |
|---|---|
| Is there transaction/event-level data underlying an entity I predict on? | GroupBy aggregation (count, mean, std, mode, recency) |
| Do I have any timestamp columns? | Extract day-of-week, hour, is_weekend, days-to-event, cyclical encoding (sin/cos of hour) |
| Do two features have a domain-meaningful ratio or product? | Interaction / ratio features (EMI/income, price/sqft) |
| Are related facts spread across multiple tables? | Merge/join carefully — verify row counts before/after |
| Do high-cardinality categoricals blow up memory or one-hot columns? | pandas Categorical dtype + target/frequency encoding |

> **🔗 Real-World Link — Feature Selection (Ride-Sharing Pricing)**
>
> 1,000 real ride-sharing bookings show exactly why feature selection needs real correlation checks, not assumptions: expected ride duration correlates strongly with price (r=0.928), while rider/driver counts barely correlate at all (r<0.04). [See the case study →](https://statso.io/feature-selection-case-study/) ·

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Calendar features from timestamps

From the `ts` timestamps build a DataFrame `feat` with columns `hour` (int) and `is_weekend` (1 for Saturday/Sunday, else 0).

In [ ]:
import pandas as pd
ts = pd.to_datetime(["2026-01-17 10:00", "2026-01-19 09:00", "2026-01-18 23:30"])
feat = None   # TODO


In [ ]:
try:
    check("hours", feat["hour"].tolist() == [10, 9, 23])
    check("weekend flags (Sat, Mon, Sun)", feat["is_weekend"].tolist() == [1, 0, 1])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
ts = pd.to_datetime(["2026-01-17 10:00", "2026-01-19 09:00", "2026-01-18 23:30"])
feat = pd.DataFrame({"hour": ts.hour, "is_weekend": (ts.dayofweek >= 5).astype(int)})

```

</details>

### Exercise 2 · Medium · Aggregate orders into customer features

Roll the order-level table up to one row per customer with columns `n_orders`, `total_spend` and `avg_order` (named aggregations), indexed by `customer_id`, into `cust`.

In [ ]:
import pandas as pd
orders = pd.DataFrame({"customer_id": ["A", "B", "A", "C", "A", "B"], "value": [100, 250, 300, 80, 200, 150]})
cust = None   # TODO


In [ ]:
try:
    check("three customers", cust is not None and len(cust) == 3)
    check("A totals", cust.loc["A", "n_orders"] == 3 and cust.loc["A", "total_spend"] == 600 and cust.loc["A", "avg_order"] == 200)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import pandas as pd
orders = pd.DataFrame({"customer_id": ["A", "B", "A", "C", "A", "B"], "value": [100, 250, 300, 80, 200, 150]})
cust = orders.groupby("customer_id").agg(n_orders=("value", "size"), total_spend=("value", "sum"), avg_order=("value", "mean"))

```

</details>

### Exercise 3 · Stretch · Interaction features

Use `PolynomialFeatures(degree=2, include_bias=False)` on the two loan features and store the generated column names in `names` and the transformed array in `Xp`.

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
X = np.array([[2, 3], [4, 5]])
names = Xp = None   # TODO (feature names: emi, income)


In [ ]:
try:
    check("five columns", Xp is not None and Xp.shape == (2, 5))
    check("names", list(names) == ["emi", "income", "emi^2", "emi income", "income^2"])
    check("interaction column", Xp[:, 3].tolist() == [6, 20])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
X = np.array([[2, 3], [4, 5]])
poly = PolynomialFeatures(degree=2, include_bias=False)
Xp = poly.fit_transform(X)
names = poly.get_feature_names_out(["emi", "income"])

```

</details>

---
*Back to the course: **Machine Learning End To End → Feature Engineering**.*